In [2]:
import os
import re
import json
import time
import random
from pathlib import Path
from typing import Dict, Any, List
from datetime import datetime, timedelta

import pandas as pd
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from threading import Semaphore, Lock

from openai import OpenAI

# =========================
# ✅ CONFIG — BEDROCK MANTLE + MISTRAL LARGE 3 675B
# =========================
BEDROCK_API_KEY = os.getenv("BEDROCK_API_KEY", "").strip()

BEDROCK_REGION = os.getenv(
    "BEDROCK_REGION",
    "ap-south-1",
).strip()

BEDROCK_BASE_URL = os.getenv(
    "BEDROCK_BASE_URL",
    f"https://bedrock-mantle.{BEDROCK_REGION}.api.aws/v1",
).strip()

BEDROCK_MODEL = os.getenv(
    "BEDROCK_MODEL",
    "google.gemma-3-27b-it",
).strip()

client = OpenAI(
    api_key=BEDROCK_API_KEY,
    base_url=BEDROCK_BASE_URL,
    timeout=180.0,
    max_retries=0,
)

# =========================
# ✅ ADVANCED RATE LIMITER WITH MULTIPROCESSING SUPPORT
# =========================
class AdvancedRateLimiter:
    """Rate limiter to handle N RPM with controlled multiprocessing.

    Bedrock account/model quotas generally allow far more than 15 RPM (often
    30-1000+ depending on model/tier), so defaults here are higher than the
    original Gemini script. Adjust max_requests/period_seconds/max_parallel
    to match your actual Bedrock quota.
    """

    def __init__(self, max_requests: int = 28, period_seconds: int = 60, max_parallel: int = 10):
        self.max_requests = max_requests
        self.period_seconds = period_seconds
        self.max_parallel = min(max_parallel, max_requests)  # Don't exceed rate limit
        self.requests = []  # Timestamps of recent requests
        self.lock = threading.Lock()
        self.semaphore = Semaphore(max_parallel)  # Control parallel executions

        # Stats
        self.total_requests = 0
        self.rate_limit_hits = 0
        self.batch_start_time = None
        self.requests_in_current_minute = 0

        # Token consumption statistics
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_consumed_tokens = 0

        print(f"⚡ Rate limiter configured: {max_parallel} parallel requests, {max_requests} RPM")

    def acquire_slot(self):
        """Acquire a parallel execution slot"""
        self.semaphore.acquire()

    def release_slot(self):
        """Release a parallel execution slot"""
        self.semaphore.release()

    def wait_if_needed(self):
        """Wait if necessary to comply with rate limits"""
        with self.lock:
            now = time.time()

            # Remove requests older than the period
            self.requests = [req_time for req_time in self.requests
                              if now - req_time < self.period_seconds]

            # Track batch timing
            if self.batch_start_time is None:
                self.batch_start_time = now
                self.requests_in_current_minute = 0

            # Check if we need to wait for next minute
            if len(self.requests) >= self.max_requests:
                self.rate_limit_hits += 1
                oldest_request = self.requests[0]
                wait_time = self.period_seconds - (now - oldest_request)

                if wait_time > 0:
                    print(f"⏰ Rate limit reached. Waiting {wait_time:.1f} seconds for next minute...")
                    time.sleep(wait_time)
                    now = time.time()
                    self.requests = []  # Reset for new minute
                    self.batch_start_time = now
                    self.requests_in_current_minute = 0

            # Add current request
            self.requests.append(now)
            self.total_requests += 1
            self.requests_in_current_minute += 1

            # Calculate and display stats
            if self.requests_in_current_minute > 0 and self.requests_in_current_minute % 5 == 0:
                time_elapsed = now - self.batch_start_time
                if time_elapsed > 0:
                    current_rpm = (self.requests_in_current_minute / time_elapsed) * 60
                    print(f"📊 Current batch: {self.requests_in_current_minute}/{self.max_requests} requests, "
                          f"{current_rpm:.1f} RPM, {time_elapsed:.1f}s elapsed")

            return now

    def record_token_usage(
        self,
        input_tokens: int,
        output_tokens: int,
        total_tokens: int
    ) -> Dict[str, int]:
        """Thread-safely record token usage and return cumulative totals."""
        with self.lock:
            self.total_input_tokens += int(input_tokens or 0)
            self.total_output_tokens += int(output_tokens or 0)
            self.total_consumed_tokens += int(total_tokens or 0)

            return {
                "total_input_tokens": self.total_input_tokens,
                "total_output_tokens": self.total_output_tokens,
                "total_consumed_tokens": self.total_consumed_tokens,
            }

    def get_status(self):
        """Get current rate limiter status"""
        with self.lock:
            now = time.time()
            recent_requests = [req_time for req_time in self.requests
                                if now - req_time < self.period_seconds]
            return {
                'recent_requests': len(recent_requests),
                'total_requests': self.total_requests,
                'rate_limit_hits': self.rate_limit_hits,
                'available_slots': self.semaphore._value,
                'total_input_tokens': self.total_input_tokens,
                'total_output_tokens': self.total_output_tokens,
                'total_consumed_tokens': self.total_consumed_tokens
            }


# Initialize rate limiter — tune to your Bedrock quota
rate_limiter = AdvancedRateLimiter(max_requests=28, period_seconds=60, max_parallel=10)

# =========================
# ✅ COMPREHENSIVE LLM EXTRACTION WITH PARALLEL RATE LIMITING (BEDROCK MISTRAL LARGE 3 675B)
# =========================
def extract_all_details_bedrock(text: str, retries: int = 3) -> Dict[str, Any]:
    """
    Extract all real estate details using Mistral Large 3 675B through Bedrock Mantle.
    Includes rate limiting for parallel requests.
    """

    prompt = f"""
You are an expert Marathi real estate document parser. Extract ALL details from the following propertydescription (property document) text.

**CRITICAL INSTRUCTIONS:**
1. Return ONLY valid JSON, no explanations
2. For project names and location details, extract BOTH English translation AND original Marathi/Hindi text
3. For other fields, translate Marathi to English but keep original names/numbers
4. For areas, extract both sq.m and sq.ft if available
5. Extract ALL mentioned details

**EXTRACT THESE FIELDS:**

1. **PROJECT/BUILDING DETAILS:**
   - project_name_en: English translation of project/building name
   - project_name_original: Original Marathi/Hindi project name as written in document
   - building_name_en: English translation if different from project name
   - building_name_original: Original Marathi/Hindi building name
   - property_type: "Flat", "Office", "Shop", "Plot", "Bungalow", "Transfer Deed", etc.
   - transaction_type: "Sale", "Purchase", "Lease", "Gift", "Inheritance", etc.

2. **PROPERTY IDENTIFICATION:**
   - flat_no: Flat/Unit number
   - floor_no: Floor number
   - wing_no: Wing number if mentioned
   - plot_no: Plot number
   - survey_no: Survey number
   - CTS_no: CTS number (extract both old and new if mentioned)
   - block_no: Block number/sector
   - society_name: Housing society name

3. **AREA DETAILS (extract all mentioned):**
   - carpet_area: In sq.m and/or sq.ft
   - builtup_area: In sq.m and/or sq.ft
   - super_builtup_area: In sq.m and/or sq.ft
   - saleable_area: In sq.m and/or sq.ft
   - terrace_area: In sq.m and/or sq.ft
   - balcony_area: In sq.m and/or sq.ft
   - total_area: In sq.m and/or sq.ft
   - plot_area: In sq.m and/or sq.ft

4. **LOCATION & ADDRESS:**
   - full_address_en: Complete address in English (translated)
   - full_address_original: Complete address in original Marathi/Hindi as written
   - road_name_en: Road/Street name in English (translated)
   - road_name_original: Road/Street name in original Marathi/Hindi
   - locality_en: Locality/Area in English (translated)
   - locality_original: Locality/Area in original Marathi/Hindi
   - city_en: City in English (translated)
   - city_original: City in original Marathi/Hindi as written
   - pincode: Pincode (numeric only)
   - taluka_en: Taluka in English (translated)
   - taluka_original: Taluka in original Marathi/Hindi
   - district_en: District in English (translated)
   - district_original: District in original Marathi/Hindi
   - state_en: State in English (translated)
   - state_original: State in original Marathi/Hindi as written

5. **OTHER DETAILS:**
   - registration_no: Registration number
   - document_date: Document date if mentioned
   - owner_name: Owner name if mentioned
   - owner_name_original: Owner name in original Marathi/Hindi script
   - remarks: Any other important information
   - remarks_original: Remarks in original Marathi/Hindi

**SPECIAL RULES FOR PROJECT NAME & LOCATION:**
1. **project_name_original**: Extract EXACTLY as written in Marathi/Hindi script (e.g., "शिवशक्ती अपार्टमेंट", "साई कृपा निवास")
2. **project_name_en**: Provide English translation (e.g., "Shivshakti Apartment", "Sai Krupa Nivas")
3. **For address fields**: Always provide both _en (English translation) and _original (exact text)
4. Keep proper nouns (names of persons, specific building names) in their original form even in English translation
5. For numbers in addresses (like plot numbers, survey numbers), keep them exactly as in original

**GENERAL RULES:**
- Return null for fields not found
- For non-name fields, translate Marathi to English
- Keep numbers exactly as in text
- For area values, include units: e.g., "48.41 sq.m (521.09 sq.ft)"
- Extract property_type from context: e.g., "सदनिका" = "Flat", "कार्यालय" = "Office"
- Extract transaction_type: "विक्री" = "Sale", "खरेदी" = "Purchase", "हस्तांतरण" = "Transfer"

**TEXT TO PARSE:**
\"\"\"{text or ''}\"\"\"

Return ONLY JSON with the above keys.
"""

    content = ""
    for attempt in range(1, retries + 1):
        try:
            # Acquire parallel execution slot
            rate_limiter.acquire_slot()

            try:
                # Apply rate limiting before making API call
                request_time = rate_limiter.wait_if_needed()

                # Log request timing info
                request_num = rate_limiter.total_requests
                if request_num % 5 == 0 or attempt == 1:
                    print(f"📤 Request #{request_num} sent at {datetime.fromtimestamp(request_time).strftime('%H:%M:%S')}")

                response = client.chat.completions.create(
                    model=BEDROCK_MODEL,
                    messages=[
                        {
                            "role": "system",
                            "content": (
                                "You are a high-precision Marathi real-estate "
                                "document extraction engine. Return only valid JSON."
                            ),
                        },
                        {"role": "user", "content": prompt},
                    ],
                    temperature=0,
                    max_tokens=4096,
                )
                content = response.choices[0].message.content.strip()

                # =========================
                # TOKEN USAGE FOR THIS ROW
                # =========================
                usage = response.usage

                prompt_tokens = usage.prompt_tokens if usage else 0
                completion_tokens = usage.completion_tokens if usage else 0
                total_tokens = usage.total_tokens if usage else 0

                # Record cumulative token usage safely across parallel threads
                cumulative_tokens = rate_limiter.record_token_usage(
                    input_tokens=prompt_tokens,
                    output_tokens=completion_tokens,
                    total_tokens=total_tokens,
                )

                # tqdm.write() prints without permanently breaking the progress bar
                tqdm.write(
                    f"🪙 Request #{request_num} tokens | "
                    f"Input: {prompt_tokens:,} | "
                    f"Output: {completion_tokens:,} | "
                    f"Total: {total_tokens:,} | "
                    f"Cumulative: "
                    f"{cumulative_tokens['total_input_tokens']:,} input + "
                    f"{cumulative_tokens['total_output_tokens']:,} output = "
                    f"{cumulative_tokens['total_consumed_tokens']:,} total"
                )

                # Clean JSON from markdown fences
                if content.startswith("```"):
                    lines = content.split("\n")
                    content = (
                        "\n".join(lines[1:-1])
                        if lines[0].startswith("```") and lines[-1].startswith("```")
                        else content[3:-3]
                    )

                # Parse JSON
                data = json.loads(content)

                # Clean and validate data
                cleaned_data = {}

                for key, value in data.items():
                    if isinstance(value, str):
                        value = value.strip()

                        if value.lower() in ["", "null", "none", "na", "n/a"]:
                            value = None

                    cleaned_data[key] = value

                # Keep token usage inside the same dictionary
                cleaned_data["_token_usage"] = {
                    "request_number": request_num,
                    "input_tokens": prompt_tokens,
                    "output_tokens": completion_tokens,
                    "total_tokens": total_tokens,
                }

                return cleaned_data


            finally:
                # Always release the slot
                rate_limiter.release_slot()

        except json.JSONDecodeError as e:
            print(f"[Attempt {attempt}] JSON parse error: {e}")
            print(f"Raw response: {content[:500]}...")

            # Try to fix common JSON issues
            try:
                content_clean = re.sub(r'^[^{]*', '', content)  # Remove before first {
                content_clean = re.sub(r'[^}]*$', '', content_clean)  # Remove after last }
                data = json.loads(content_clean)
                return data
            except Exception:
                pass

        except Exception as e:
            print(f"[Attempt {attempt}] Error: {e}")

        # Exponential backoff with jitter
        if attempt < retries:
            wait_time = min(60, 2 ** attempt + random.randint(1, 10))
            print(f"⏳ Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)

    # Return empty structure if all retries fail
    return get_empty_structure()


def get_empty_structure() -> Dict[str, Any]:
    """Return empty structure with all expected keys"""
    return {
        # Project/Building - Bilingual
        "project_name_en": None,
        "project_name_original": None,
        "building_name_en": None,
        "building_name_original": None,
        "property_type": None,
        "transaction_type": None,

        # Property Identification
        "flat_no": None,
        "floor_no": None,
        "wing_no": None,
        "plot_no": None,
        "survey_no": None,
        "CTS_no": None,
        "block_no": None,
        "society_name": None,

        # Area Details
        "carpet_area": None,
        "builtup_area": None,
        "super_builtup_area": None,
        "saleable_area": None,
        "terrace_area": None,
        "balcony_area": None,
        "total_area": None,
        "plot_area": None,

        # Location & Address - Bilingual
        "full_address_en": None,
        "full_address_original": None,
        "road_name_en": None,
        "road_name_original": None,
        "locality_en": None,
        "locality_original": None,
        "city_en": None,
        "city_original": None,
        "pincode": None,
        "taluka_en": None,
        "taluka_original": None,
        "district_en": None,
        "district_original": None,
        "state_en": None,
        "state_original": None,

        # Other Details
        "registration_no": None,
        "document_date": None,
        "owner_name": None,
        "owner_name_original": None,
        "remarks": None,
        "remarks_original": None
    }

# =========================
# ✅ PARALLEL PROCESSING WITH CONTROLLED MULTIPROCESSING
# =========================
def parallel_extract_bedrock(df_chunk: pd.DataFrame, max_workers: int = 10) -> pd.DataFrame:
    """
    Process rows in parallel with controlled multiprocessing
    """
    total_items = len(df_chunk)

    print(f"🚀 Processing {total_items} items with {max_workers} parallel workers")

    def process_row(row_idx, row):
        text = row.get("propertydescription", "")
        if not isinstance(text, str):
            text = ""
        return row_idx, extract_all_details_bedrock(text)

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {}
        for idx, (_, row) in enumerate(df_chunk.iterrows()):
            future = executor.submit(process_row, idx, row)
            futures[future] = idx

        # Process results as they complete
        results = [None] * total_items  # Pre-allocate list

        with tqdm(total=total_items, desc="Processing") as pbar:
            for future in as_completed(futures):
                try:
                    row_idx, result = future.result(timeout=300)  # 5 minute timeout
                    results[row_idx] = result

                    # Update progress with rate limiter stats
                    status = rate_limiter.get_status()
                    pbar.set_postfix({
                        'RPM': f"{status['recent_requests']}/{rate_limiter.max_requests}",
                        'Requests': status['total_requests'],
                        'Tokens': f"{status['total_consumed_tokens']:,}"
                    })
                    pbar.update(1)

                except Exception as e:
                    print(f"\n[Error] {e}")
                    row_idx = futures[future]
                    results[row_idx] = get_empty_structure()
                    pbar.update(1)

    # Keep complete LLM output as one dictionary per row
    combined_df = df_chunk.reset_index(drop=True).copy()
    combined_df["llm_output"] = results

    return combined_df


# =========================
# ✅ TOKEN LEDGER — ONE EXCEL FILE PER CHUNK
# =========================
def save_chunk_token_ledger(
    result_df: pd.DataFrame,
    chunk_idx: int,
    output_dir: str,
    source_start_row: int = 0
) -> Path:
    """
    Create one token ledger workbook for a processed chunk.

    Workbook sheets:
      1. Token_Details  -> one row per source record/request
      2. Chunk_Summary  -> total and average token usage for the chunk
    """
    ledger_rows = []

    for row_in_chunk, llm_output in enumerate(result_df.get("llm_output", []), start=1):
        token_usage = {}

        if isinstance(llm_output, dict):
            token_usage = llm_output.get("_token_usage", {}) or {}

        input_tokens = int(token_usage.get("input_tokens", 0) or 0)
        output_tokens = int(token_usage.get("output_tokens", 0) or 0)
        total_tokens = int(token_usage.get("total_tokens", 0) or 0)
        request_number = token_usage.get("request_number")

        ledger_rows.append({
            "chunk_number": chunk_idx,
            "row_in_chunk": row_in_chunk,
            "source_excel_row": source_start_row + row_in_chunk + 1,
            "request_number": request_number,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": total_tokens,
            "status": "Success" if total_tokens > 0 else "No token data",
        })

    details_df = pd.DataFrame(ledger_rows)

    if details_df.empty:
        details_df = pd.DataFrame(columns=[
            "chunk_number",
            "row_in_chunk",
            "source_excel_row",
            "request_number",
            "input_tokens",
            "output_tokens",
            "total_tokens",
            "status",
        ])

    successful_rows = int((details_df["total_tokens"] > 0).sum()) if len(details_df) else 0
    failed_rows = int((details_df["total_tokens"] <= 0).sum()) if len(details_df) else 0

    total_input = int(details_df["input_tokens"].sum()) if len(details_df) else 0
    total_output = int(details_df["output_tokens"].sum()) if len(details_df) else 0
    total_combined = int(details_df["total_tokens"].sum()) if len(details_df) else 0

    summary_df = pd.DataFrame([{
        "chunk_number": chunk_idx,
        "records_in_chunk": len(details_df),
        "successful_records": successful_rows,
        "records_without_token_data": failed_rows,
        "total_input_tokens": total_input,
        "total_output_tokens": total_output,
        "total_tokens": total_combined,
        "average_input_tokens_per_record": round(
            total_input / successful_rows, 2
        ) if successful_rows else 0,
        "average_output_tokens_per_record": round(
            total_output / successful_rows, 2
        ) if successful_rows else 0,
        "average_total_tokens_per_record": round(
            total_combined / successful_rows, 2
        ) if successful_rows else 0,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])

    ledger_path = Path(output_dir) / f"token_ledger_chunk_{chunk_idx}.xlsx"

    with pd.ExcelWriter(ledger_path, engine="openpyxl") as writer:
        details_df.to_excel(writer, sheet_name="Token_Details", index=False)
        summary_df.to_excel(writer, sheet_name="Chunk_Summary", index=False)

    print(f"🧾 Token ledger saved: {ledger_path}")
    print(
        f"   Chunk {chunk_idx} tokens | "
        f"Input: {total_input:,} | "
        f"Output: {total_output:,} | "
        f"Total: {total_combined:,}"
    )

    return ledger_path


def merge_token_ledgers(
    output_dir: str,
    final_ledger_path: str = "token_ledger_all_chunks.xlsx"
) -> pd.DataFrame:
    """Merge all per-chunk token ledgers into one final ledger workbook."""
    ledger_files = sorted(
        Path(output_dir).glob("token_ledger_chunk_*.xlsx"),
        key=lambda path: int(
            re.search(r"token_ledger_chunk_(\d+)\.xlsx", path.name).group(1)
        )
    )

    if not ledger_files:
        print(f"⚠️ No token ledger files found in {output_dir}")
        return pd.DataFrame()

    detail_frames = []

    for ledger_file in ledger_files:
        detail_frames.append(
            pd.read_excel(ledger_file, sheet_name="Token_Details")
        )

    all_details = pd.concat(detail_frames, ignore_index=True)

    per_chunk_summary = (
        all_details.groupby("chunk_number", as_index=False)
        .agg(
            records_in_chunk=("row_in_chunk", "count"),
            successful_records=("total_tokens", lambda s: int((s > 0).sum())),
            total_input_tokens=("input_tokens", "sum"),
            total_output_tokens=("output_tokens", "sum"),
            total_tokens=("total_tokens", "sum"),
        )
    )

    per_chunk_summary["average_total_tokens_per_record"] = (
        per_chunk_summary["total_tokens"]
        / per_chunk_summary["successful_records"].replace(0, pd.NA)
    ).fillna(0).round(2)

    grand_summary = pd.DataFrame([{
        "total_chunks": len(per_chunk_summary),
        "total_records": len(all_details),
        "successful_records": int((all_details["total_tokens"] > 0).sum()),
        "total_input_tokens": int(all_details["input_tokens"].sum()),
        "total_output_tokens": int(all_details["output_tokens"].sum()),
        "total_tokens": int(all_details["total_tokens"].sum()),
        "average_total_tokens_per_successful_record": round(
            all_details.loc[
                all_details["total_tokens"] > 0, "total_tokens"
            ].mean(),
            2,
        ) if (all_details["total_tokens"] > 0).any() else 0,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }])

    with pd.ExcelWriter(final_ledger_path, engine="openpyxl") as writer:
        all_details.to_excel(writer, sheet_name="All_Token_Details", index=False)
        per_chunk_summary.to_excel(writer, sheet_name="Per_Chunk_Summary", index=False)
        grand_summary.to_excel(writer, sheet_name="Grand_Summary", index=False)

    print(f"✅ Combined token ledger saved: {final_ledger_path}")

    return all_details


# =========================
# ✅ BATCH PROCESSOR WITH INTELLIGENT RATE MANAGEMENT
# =========================
def process_chunk_with_intelligent_rate_limits(chunk_df: pd.DataFrame, chunk_idx: int, total_chunks: int) -> pd.DataFrame:
    """
    Process a single chunk with intelligent rate management
    """
    chunk_size = len(chunk_df)
    max_workers = min(rate_limiter.max_parallel, chunk_size) if chunk_size > 0 else 1

    print(f"\n{'='*60}")
    print(f"📦 Processing Chunk {chunk_idx}/{total_chunks}")
    print(f"{'='*60}")
    print(f"📊 Chunk size: {chunk_size} records")
    print(f"⚡ Parallel workers: {max_workers}")

    start_time = time.time()

    try:
        result_df = parallel_extract_bedrock(chunk_df, max_workers=max_workers)

        end_time = time.time()
        elapsed = end_time - start_time

        actual_rpm = (chunk_size / elapsed) * 60 if elapsed > 0 else 0

        print(f"\n✅ Chunk {chunk_idx} completed in {elapsed:.1f} seconds")
        print(f"📈 Performance: {actual_rpm:.1f} RPM (target: ≤{rate_limiter.max_requests} RPM)")
        print(f"📊 Rate limiter stats:")
        print(f"   Total requests: {rate_limiter.total_requests}")
        print(f"   Rate limit hits: {rate_limiter.rate_limit_hits}")
        print(f"   Input tokens: {rate_limiter.total_input_tokens:,}")
        print(f"   Output tokens: {rate_limiter.total_output_tokens:,}")
        print(f"   Total tokens: {rate_limiter.total_consumed_tokens:,}")

        return result_df

    except Exception as e:
        print(f"\n❌ Error processing chunk {chunk_idx}: {e}")
        raise

# =========================
# ✅ MAIN EXECUTION FLOW WITH SMART BATCHING
# =========================
def run_extraction_pipeline(
    input_csv_path: str,
    output_dir: str = "output",
    chunk_size: int = 60,
    resume: bool = True
):
    """
    Main pipeline for extracting real estate details with intelligent rate limiting
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Load data
    try:
        if input_csv_path.endswith('.xlsx'):
            df = pd.read_excel(input_csv_path)
        else:
            df = pd.read_csv(input_csv_path)
    except Exception as e:
        raise ValueError(f"Error loading input file: {e}")

    if "propertydescription" not in df.columns:
        raise ValueError(f"'propertydescription' column not found. Available: {list(df.columns)}")

    # Get existing chunks if resuming
    existing_chunks = set()
    if resume:
        for file in Path(output_dir).glob("extracted_chunk_*.xlsx"):
            match = re.search(r"extracted_chunk_(\d+)\.xlsx", file.name)
            if match:
                existing_chunks.add(int(match.group(1)))

    total_rows = len(df)
    total_chunks = (total_rows + chunk_size - 1) // chunk_size

    batches_needed = (total_rows + rate_limiter.max_requests - 1) // rate_limiter.max_requests
    estimated_time_minutes = batches_needed
    estimated_time_hours = estimated_time_minutes / 60

    print(f"\n{'='*60}")
    print(f"🚀 EXTRACTION JOB SUMMARY (BEDROCK MISTRAL LARGE 3 675B)")
    print(f"{'='*60}")
    print(f"🤖 Model: {BEDROCK_MODEL}")
    print(f"📊 Total records: {total_rows:,}")
    print(f"📦 Chunk size: {chunk_size}")
    print(f"📁 Total chunks: {total_chunks}")
    print(f"⚡ Max parallel requests: {rate_limiter.max_parallel}")
    print(f"⏱️ Rate limit: {rate_limiter.max_requests} requests per minute")
    print(f"📈 Estimated batches: {batches_needed}")
    print(f"🕐 Estimated time: {estimated_time_minutes:.1f} minutes ({estimated_time_hours:.1f} hours)")
    print(f"{'='*60}\n")

    for chunk_idx in range(1, total_chunks + 1):
        output_file = Path(output_dir) / f"extracted_chunk_{chunk_idx}.xlsx"

        if resume and chunk_idx in existing_chunks:
            print(f"✓ Chunk {chunk_idx}/{total_chunks} already processed (skipping)")
            continue

        start_idx = (chunk_idx - 1) * chunk_size
        end_idx = min(chunk_idx * chunk_size, total_rows)
        chunk_df = df.iloc[start_idx:end_idx].copy()

        try:
            print(f"\n📂 Loading chunk {chunk_idx} (rows {start_idx:,}-{end_idx-1:,})...")

            result_df = process_chunk_with_intelligent_rate_limits(
                chunk_df, chunk_idx, total_chunks
            )

            result_df.to_excel(output_file, index=False)
            print(f"💾 Saved chunk {chunk_idx} to {output_file}")

            # Save one token ledger workbook for this chunk
            save_chunk_token_ledger(
                result_df=result_df,
                chunk_idx=chunk_idx,
                output_dir=output_dir,
                source_start_row=start_idx,
            )

        except Exception as e:
            print(f"✗ Error processing chunk {chunk_idx}: {e}")
            empty_results = pd.DataFrame([get_empty_structure() for _ in range(len(chunk_df))])
            combined_df = pd.concat([chunk_df.reset_index(drop=True), empty_results], axis=1)
            combined_df.to_excel(output_file, index=False)
            print(f"⚠️ Saved empty results for chunk {chunk_idx}")

        if chunk_idx < total_chunks:
            pause_time = 2
            print(f"⏸️ Pausing for {pause_time} seconds between chunks...")
            time.sleep(pause_time)

    print(f"\n{'='*60}")
    print("🎉 EXTRACTION COMPLETE!")
    print(f"{'='*60}")
    print(f"📊 FINAL STATISTICS:")
    print(f"   Total requests made: {rate_limiter.total_requests:,}")
    print(f"   Rate limit hits: {rate_limiter.rate_limit_hits}")
    print(f"   Input tokens consumed: {rate_limiter.total_input_tokens:,}")
    print(f"   Output tokens consumed: {rate_limiter.total_output_tokens:,}")
    print(f"   Total tokens consumed: {rate_limiter.total_consumed_tokens:,}")
    print(f"   Total records processed: {total_rows:,}")
    print(f"{'='*60}")

# =========================
# ✅ MERGE RESULTS
# =========================
def merge_results(output_dir: str, final_output_path: str = "final_output.xlsx"):
    """Merge all chunk files into final Excel"""

    chunk_files = sorted(
        Path(output_dir).glob("extracted_chunk_*.xlsx"),
        key=lambda x: int(re.search(r"extracted_chunk_(\d+)\.xlsx", x.name).group(1))
    )

    if not chunk_files:
        raise ValueError(f"No chunk files found in {output_dir}")

    print(f"\n📂 Merging {len(chunk_files)} chunk files...")

    all_dfs = []
    for file in tqdm(chunk_files, desc="Loading chunks"):
        df = pd.read_excel(file)
        all_dfs.append(df)

    final_df = pd.concat(all_dfs, ignore_index=True)

    final_df.to_excel(final_output_path, index=False)
    print(f"✅ Merged {len(chunk_files)} chunks into {final_output_path}")
    print(f"📊 Total records: {len(final_df):,}")

    return final_df

# =========================
# ✅ ANALYSIS & VALIDATION
# =========================
def analyze_results(df: pd.DataFrame):
    """Analyze extraction results"""

    print("\n" + "="*60)
    print("📈 EXTRACTION ANALYSIS")
    print("="*60)

    key_fields = [
        "project_name_en", "property_type", "flat_no",
        "carpet_area", "full_address", "block_no"
    ]

    for field in key_fields:
        if field in df.columns:
            non_null = df[field].notna().sum()
            percentage = (non_null / len(df)) * 100
            print(f"{field:25s}: {non_null:5d} / {len(df):5d} ({percentage:6.1f}%)")

    if "property_type" in df.columns:
        print(f"\n{'='*60}")
        print("🏠 PROPERTY TYPE DISTRIBUTION")
        print(f"{'='*60}")
        prop_types = df["property_type"].value_counts(dropna=False).head(10)
        for prop_type, count in prop_types.items():
            percentage = (count / len(df)) * 100
            prop_type_display = "Unknown" if pd.isna(prop_type) else prop_type
            print(f"{prop_type_display:25s}: {count:5d} ({percentage:6.1f}%)")

# =========================
# ✅ RESUME FROM SPECIFIC CHUNK
# =========================
def resume_from_chunk(start_chunk: int, input_csv_path: str, output_dir: str = "output"):
    """
    Resume processing from a specific chunk number
    """
    if input_csv_path.endswith('.xlsx'):
        df = pd.read_excel(input_csv_path)
    else:
        df = pd.read_csv(input_csv_path)

    total_rows = len(df)
    chunk_size = 60  # Must match original chunk_size
    total_chunks = (total_rows + chunk_size - 1) // chunk_size

    print(f"🔁 Resuming from chunk {start_chunk}/{total_chunks}")

    for chunk_idx in range(start_chunk, total_chunks + 1):
        output_file = Path(output_dir) / f"extracted_chunk_{chunk_idx}.xlsx"

        if output_file.exists():
            print(f"✓ Chunk {chunk_idx}/{total_chunks} already exists (skipping)")
            continue

        start_idx = (chunk_idx - 1) * chunk_size
        end_idx = min(chunk_idx * chunk_size, total_rows)
        chunk_df = df.iloc[start_idx:end_idx].copy()

        try:
            result_df = process_chunk_with_intelligent_rate_limits(chunk_df, chunk_idx, total_chunks)
            result_df.to_excel(output_file, index=False)
            print(f"💾 Saved chunk {chunk_idx} to {output_file}")

            save_chunk_token_ledger(
                result_df=result_df,
                chunk_idx=chunk_idx,
                output_dir=output_dir,
                source_start_row=start_idx,
            )

        except Exception as e:
            print(f"✗ Error processing chunk {chunk_idx}: {e}")
            empty_results = pd.DataFrame([get_empty_structure() for _ in range(len(chunk_df))])
            combined_df = pd.concat([chunk_df.reset_index(drop=True), empty_results], axis=1)
            combined_df.to_excel(output_file, index=False)
            print(f"⚠️ Saved empty results for chunk {chunk_idx}")

        if chunk_idx < total_chunks:
            print("⏸️ Pausing for 2 seconds between chunks...")
            time.sleep(2)

# =========================
# ✅ MAIN EXECUTION
# =========================
if __name__ == "__main__":
    # Configuration
    INPUT_FILE = r"D:\Nilesh\Task\Sogaon\Sagaon_Processed_igr_data.xlsx"
    OUTPUT_DIR = "extracted_results_parallel"
    FINAL_OUTPUT = r"D:\Nilesh\Task\Sogaon\Sagaon_Processed_igr_data_output.xlsx"

    if not BEDROCK_API_KEY:
        raise SystemExit(
            "❌ BEDROCK_API_KEY is missing. "
            "Set it in PowerShell using: "
            '$env:BEDROCK_API_KEY="YOUR_NEW_BEDROCK_API_KEY"'
        )

    try:
        print("\n" + "="*70)
        print("🚀 REAL ESTATE DATA EXTRACTION WITH PARALLEL PROCESSING (BEDROCK MISTRAL LARGE 3 675B)")
        print("="*70)
        print("⚡ Features:")
        print(f"   • Model: {BEDROCK_MODEL}")
        print(f"   • {rate_limiter.max_parallel} parallel requests per batch")
        print(f"   • Intelligent {rate_limiter.max_requests} RPM rate limiting")
        print("   • Automatic batch management")
        print("   • Progress tracking with statistics")
        print("="*70)

        print("\n📂 Loading input file...")
        run_extraction_pipeline(
            input_csv_path=INPUT_FILE,
            output_dir=OUTPUT_DIR,
            chunk_size=60,
            resume=True
        )

        print("\n📦 Merging results...")
        final_df = merge_results(OUTPUT_DIR, FINAL_OUTPUT)

        print("\n🧾 Merging token ledgers...")
        merge_token_ledgers(
            output_dir=OUTPUT_DIR,
            final_ledger_path="token_ledger_all_chunks.xlsx",
        )

        analyze_results(final_df)

        print(f"\n✅ EXTRACTION COMPLETE!")
        print(f"📁 Results saved to: {FINAL_OUTPUT}")
        print(f"📊 Total records extracted: {len(final_df):,}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Process interrupted by user")
        print(f"📂 Partial results saved in: {OUTPUT_DIR}/")
        print("🔁 Use resume_from_chunk() function to continue where you left off.")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

⚡ Rate limiter configured: 10 parallel requests, 28 RPM

🚀 REAL ESTATE DATA EXTRACTION WITH PARALLEL PROCESSING (BEDROCK MISTRAL LARGE 3 675B)
⚡ Features:
   • Model: google.gemma-3-27b-it
   • 10 parallel requests per batch
   • Intelligent 28 RPM rate limiting
   • Automatic batch management
   • Progress tracking with statistics

📂 Loading input file...

🚀 EXTRACTION JOB SUMMARY (BEDROCK MISTRAL LARGE 3 675B)
🤖 Model: google.gemma-3-27b-it
📊 Total records: 468
📦 Chunk size: 60
📁 Total chunks: 8
⚡ Max parallel requests: 10
⏱️ Rate limit: 28 requests per minute
📈 Estimated batches: 17
🕐 Estimated time: 17.0 minutes (0.3 hours)


📂 Loading chunk 1 (rows 0-59)...

📦 Processing Chunk 1/8
📊 Chunk size: 60 records
⚡ Parallel workers: 10
🚀 Processing 60 items with 10 parallel workers
📤 Request #1 sent at 12:16:17
📤 Request #2 sent at 12:16:17
📤 Request #3 sent at 12:16:17
📤 Request #4 sent at 12:16:17
📊 Current batch: 5/28 requests, 160516.8 RPM, 0.0s elapsed
📤 Request #5 sent at 12:16:17
📤

Processing:   3%|▎         | 2/60 [00:08<08:17,  8.57s/it, RPM=13/28, Requests=13, Tokens=5,173]

🪙 Request #4 tokens | Input: 1,201 | Output: 560 | Total: 1,761 | Cumulative: 1,201 input + 560 output = 1,761 total
📤 Request #11 sent at 12:16:26
🪙 Request #6 tokens | Input: 1,136 | Output: 490 | Total: 1,626 | Cumulative: 2,337 input + 1,050 output = 3,387 total
📤 Request #12 sent at 12:16:26
🪙 Request #2 tokens | Input: 1,212 | Output: 574 | Total: 1,786 | Cumulative: 3,549 input + 1,624 output = 5,173 total
📤 Request #13 sent at 12:16:26


Processing:   7%|▋         | 4/60 [00:09<01:47,  1.91s/it, RPM=14/28, Requests=14, Tokens=6,945]

🪙 Request #8 tokens | Input: 1,198 | Output: 574 | Total: 1,772 | Cumulative: 4,747 input + 2,198 output = 6,945 total
📤 Request #14 sent at 12:16:27


...

🪙 Request #468 tokens | Input: 1,129 | Output: 474 | Total: 1,603 | Cumulative: 576,796 input + 306,323 output = 883,119 total


Processing: 100%|██████████| 48/48 [01:44<00:00,  2.17s/it, RPM=20/28, Requests=468, Tokens=887,045]


🪙 Request #463 tokens | Input: 1,226 | Output: 718 | Total: 1,944 | Cumulative: 578,022 input + 307,041 output = 885,063 total
🪙 Request #465 tokens | Input: 1,237 | Output: 745 | Total: 1,982 | Cumulative: 579,259 input + 307,786 output = 887,045 total

✅ Chunk 8 completed in 104.3 seconds
📈 Performance: 27.6 RPM (target: ≤28 RPM)
📊 Rate limiter stats:
   Total requests: 468
   Rate limit hits: 16
   Input tokens: 579,259
   Output tokens: 307,786
   Total tokens: 887,045
💾 Saved chunk 8 to extracted_results_parallel\extracted_chunk_8.xlsx
🧾 Token ledger saved: extracted_results_parallel\token_ledger_chunk_8.xlsx
   Chunk 8 tokens | Input: 58,782 | Output: 30,480 | Total: 89,262

🎉 EXTRACTION COMPLETE!
📊 FINAL STATISTICS:
   Total requests made: 468
   Rate limit hits: 16
   Input tokens consumed: 579,259
   Output tokens consumed: 307,786
   Total tokens consumed: 887,045
   Total records processed: 468

📦 Merging results...

📂 Merging 8 chunk files...


Loading chunks: 100%|██████████| 8/8 [00:00<00:00, 36.33it/s]


✅ Merged 8 chunks into D:\Nilesh\Task\Sogaon\Sagaon_Processed_igr_data_output.xlsx
📊 Total records: 468

🧾 Merging token ledgers...
✅ Combined token ledger saved: token_ledger_all_chunks.xlsx

📈 EXTRACTION ANALYSIS
project_name_en          :     8 /   468 (   1.7%)

✅ EXTRACTION COMPLETE!
📁 Results saved to: D:\Nilesh\Task\Sogaon\Sagaon_Processed_igr_data_output.xlsx
📊 Total records extracted: 468
